# CS-781 Phase 1 — Training & submission (notebook B, based on the starter notebook)

**Kaggle settings:** Accelerator **GPU T4 ×2** · Internet **On** · secret `Capstone_Phase_1` on · **Add Input → Your Work → Notebooks → `phase1-data-prep`** (notebook A).

**How to use**
1. Interactive check (≈10 min GPU): run cells 1–2, B3, B4, B5 (`models.py`, your code), B6. All four architectures must print **OK**, with an initial loss close to ln(V). Then stop the session.
2. Session 1: `SESSION_ARCHS = ["decoder", "prefix"]` in B3 → **Save Version → Save & Run All** (≈5.5 h, runs in the background).
3. Download `runs/<arch>/submission_<arch>.zip` from that version's Output tab → upload to Codabench.
4. Session 2: `SESSION_ARCHS = ["encdec", "swa"]` → Save & Run All again → submit.
5. Keep both versions: their `runs/*/best.pt` checkpoints are your foundation models for Phase 2.

## 1. Hugging Face Login

In [ ]:
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("Capstone_Phase_1")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

print("Hugging Face login successful.")

## 2. Check GPU

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

## B3. Configuration
Training and decoding values are the **starter notebook's defaults** (adapted to 1024-token blocks). Model sizes are the starter's too and will probably come out **under 50M** with your vocabulary: B6 tells you, and the sizes are **your** decision. The starter's decoding (`temperature=0.1`) produced the repetition loops in the baseline, so choose your own decoding settings.

In [ ]:
import os
import json
import shutil

from huggingface_hub import hf_hub_download


# ============================================================
# SESSION ARCHITECTURES
# ============================================================

SESSION_ARCHS = ["decoder", "prefix"]
# Session 2: change to ["encdec", "swa"]

ALL_ARCHS = ["decoder", "prefix", "encdec", "swa"]


# ============================================================
# CONFIGURATION
# ============================================================

CONFIG = {

    "seed": 42,

    # Training wall-clock time per model.
    # The two GPUs run the two models in parallel.
    "hours_per_model": 4.5,


    # ========================================================
    # TRAINING SETTINGS
    # ========================================================

    "train": {
        "block_size": 1024,
        "batch_size": 8,
        "grad_accum": 2,

        "lr": 3e-4,
        "min_lr": 0.0,
        "warmup_steps": 500,

        "weight_decay": 0.1,
        "betas": [0.9, 0.999],

        "grad_clip": 1.0,

        "log_every": 50,
        "eval_every": 500,
    },


    # ========================================================
    # MODEL SETTINGS
    # ========================================================
    #
    # d_model = 672
    # n_heads = 8
    # head dimension = 672 / 8 = 84
    #
    # All four models use the same d_model and n_heads
    # to keep the comparison fair.
    #
    # max_len remains 1024 for decoder/prefix/swa.
    # EncDec keeps 768 source + 384 target.
    #

    "models": {

        # ----------------------------------------------------
        # MODEL 1: Decoder-only / GPT-style
        # ----------------------------------------------------

        "decoder": {
            "d_model": 672,
            "n_layers": 8,
            "n_heads": 8,
            "max_len": 1024,
            "dropout": 0.1,
        },


        # ----------------------------------------------------
        # MODEL 2: Prefix-LM
        # ----------------------------------------------------

        "prefix": {
            "d_model": 672,
            "n_layers": 8,
            "n_heads": 8,
            "max_len": 1024,
            "dropout": 0.1,
        },


        # ----------------------------------------------------
        # MODEL 3: Encoder-Decoder
        # ----------------------------------------------------

        "encdec": {
            "d_model": 672,
            "n_enc_layers": 4,
            "n_dec_layers": 4,
            "n_heads": 8,
            "max_len": 768,
            "max_tgt_len": 384,
            "dropout": 0.1,
        },


        # ----------------------------------------------------
        # MODEL 4: Sliding-Window Attention + ALiBi
        # ----------------------------------------------------

        "swa": {
            "d_model": 672,
            "n_layers": 8,
            "n_heads": 8,
            "window": 256,
            "max_len": 1024,
            "dropout": 0.1,
        },
    },


    # ========================================================
    # GENERATION SETTINGS
    # ========================================================

    "gen": {

        # Avoid the highly repetitive behaviour of temperature=0.1
        "temperature": 0.7,

        # 0 = disabled
        "top_k": 0,

        # Nucleus sampling
        "top_p": 0.90,

        # Mild repetition penalty
        "rep_penalty": 1.10,

        # Keep these unchanged as instructed
        "target_words": 128,
        "max_new": 320,
        "batch_size": 16,
    },
}


# ============================================================
# WRITE CONFIGURATION FILE
# ============================================================

config_path = "/kaggle/working/run_config.json"

with open(config_path, "w") as f:
    json.dump(CONFIG, f, indent=1)


# ============================================================
# DOWNLOAD TEST SET FROM HUGGING FACE
# ============================================================

test_src = hf_hub_download(
    repo_id="Exploration-Lab/CS-781-Capstone",
    filename="Phase_1_Pretraining/Test/test.json",
    repo_type="dataset",
    token=HF_TOKEN,
)

shutil.copy(
    test_src,
    "/kaggle/working/test.json"
)


# ============================================================
# VERIFY TEST SET
# ============================================================

with open("/kaggle/working/test.json") as f:
    test_data = json.load(f)


# ============================================================
# SUMMARY
# ============================================================

print("Config written to:", config_path)

print("Session architectures:")
print(SESSION_ARCHS)

print("\nTest prompts:", len(test_data))

print("\nModel configurations:")

for arch in ALL_ARCHS:
    print(f"\n{arch}:")
    for key, value in CONFIG["models"][arch].items():
        print(f"  {key}: {value}")

print("\nGeneration configuration:")
print(json.dumps(CONFIG["gen"], indent=2))

## B4. Shared pipeline code (tokenizer wrapper, batches, losses, generation, submission files)

In [ ]:
%%writefile lm_common.py
import glob, json, os, random, sys, zipfile
import numpy as np, torch, torch.nn.functional as F

def find_input(name):
    for root in ("/kaggle/working", "/kaggle/input"):
        hits = sorted(glob.glob(f"{root}/**/{name}", recursive=True))
        if hits:
            return hits[0]
    raise FileNotFoundError(f"{name} not found — is the phase1-data-prep output added as input?")

sys.path.insert(0, os.path.dirname(find_input("bpe_tok.py")))
import bpe_tok

WORD_MARK = "".join(chr(b) for b in "\u2581".encode("utf-8"))   # how '▁' looks inside your byte-level tokens

class Tok:
    def __init__(self, path=None):
        self.bpe = bpe_tok.load_bpe(path or find_input("bpe.json"))
        self.enc = bpe_tok.FastEncoder(self.bpe)
        self.itos, self.V = self.bpe.itos, len(self.bpe.itos)
        self.pad, self.unk, self.bos, self.eos = (self.bpe.stoi[s] for s in ("<pad>", "<unk>", "<s>", "</s>"))
        self.word_start = torch.tensor([t.startswith(WORD_MARK) for t in self.itos])
    def encode(self, text):
        return self.enc.encode(text)
    def decode(self, ids):
        return bpe_tok.detokenize([self.itos[i] for i in ids if i >= 4])

def get_batch(data, batch_size, block, device, rng=np.random):
    ix = rng.randint(0, len(data) - block - 1, batch_size)
    x = torch.from_numpy(np.stack([data[i:i + block + 1] for i in ix]).astype(np.int64))
    if str(device).startswith("cuda"):
        x = x.pin_memory()
    return x.to(device, non_blocking=True)

def compute_loss(model, seq, bos_id, rng=random):
    """seq: (B, block+1). Next-token cross-entropy with the objective that matches model.kind."""
    if model.kind == "causal":
        logits, target = model(seq[:, :-1]), seq[:, 1:]
    elif model.kind == "prefix":
        T = seq.size(1) - 1
        P = rng.randint(T // 4, 3 * T // 4)                       # random prompt/continuation split
        logits = model(seq[:, :-1], prefix_len=P)[:, P - 1:]
        target = seq[:, P:]
    elif model.kind == "encdec":
        L = seq.size(1)
        S = rng.randint(max(16, L - model.max_tgt_len), min(model.max_len, L - 16))
        src, tgt = seq[:, :S], seq[:, S:]
        tgt_in = torch.cat([torch.full_like(tgt[:, :1], bos_id), tgt[:, :-1]], 1)
        logits, target = model(src, tgt_in), tgt
    else:
        raise ValueError(model.kind)
    return F.cross_entropy(logits.reshape(-1, logits.size(-1)).float(), target.reshape(-1))

@torch.no_grad()
def eval_loss(model, data, batch_size, block, device, bos_id, n_batches=20):
    model.eval()
    rng, prng, losses = np.random.RandomState(1234), random.Random(1234), []
    for _ in range(n_batches):
        seq = get_batch(data, batch_size, block, device, rng)
        with torch.autocast("cuda", dtype=torch.float16):
            losses.append(compute_loss(model, seq, bos_id, prng).item())
    model.train()
    return float(np.mean(losses))

def sample_next(logits, generated, temperature, top_k, top_p, rep_penalty, banned):
    logits = logits.float().clone()
    logits[:, banned] = -float("inf")
    if rep_penalty != 1.0 and generated.size(1) > 0:
        prev = torch.gather(logits, 1, generated)
        logits.scatter_(1, generated, torch.where(prev > 0, prev / rep_penalty, prev * rep_penalty))
    if temperature <= 0:
        return logits.argmax(-1)
    logits = logits / temperature
    if top_k and top_k < logits.size(-1):
        kth = torch.topk(logits, top_k).values[:, -1:]
        logits = logits.masked_fill(logits < kth, -float("inf"))
    if top_p < 1.0:
        sorted_logits, idx = torch.sort(logits, descending=True)
        probs = sorted_logits.softmax(-1)
        sorted_logits = sorted_logits.masked_fill(probs.cumsum(-1) - probs > top_p, -float("inf"))
        logits = torch.full_like(logits, -float("inf")).scatter(1, idx, sorted_logits)
    return torch.multinomial(logits.softmax(-1), 1).squeeze(1)

@torch.no_grad()
def generate(model, tok, prompts, device, temperature=1.0, top_k=0, top_p=1.0, rep_penalty=1.0,
             target_words=128, max_new=320, batch_size=16, seed=0, log_every=10):
    """Continue each prompt until it has `target_words` words. Prompts are sorted by length and each
    batch is left-truncated to its shortest prompt, so no padding is needed (we keep the END of the prompt)."""
    torch.manual_seed(seed)
    model.eval()
    ids = [tok.encode(p) for p in prompts]
    order = sorted(range(len(ids)), key=lambda i: len(ids[i]))
    ws = tok.word_start.to(device)
    banned = torch.tensor([tok.pad, tok.unk, tok.bos, tok.eos], device=device)
    if model.kind == "encdec":
        max_prompt, max_new = model.max_len, min(max_new, model.max_tgt_len - 1)
    else:
        max_prompt = model.max_len - max_new
    out = [""] * len(ids)
    for bi, b0 in enumerate(range(0, len(order), batch_size)):
        bidx = order[b0:b0 + batch_size]
        L = min(min(len(ids[i]) for i in bidx), max_prompt)
        prompt = torch.tensor([ids[i][-L:] for i in bidx], device=device)
        B = len(bidx)
        gen = torch.empty(B, 0, dtype=torch.long, device=device)
        for _ in range(max_new):
            with torch.autocast("cuda", dtype=torch.float16):
                if model.kind == "encdec":
                    tgt = torch.cat([torch.full((B, 1), tok.bos, device=device), gen], 1)
                    logits = model(prompt, tgt)[:, -1]
                elif model.kind == "prefix":
                    logits = model(torch.cat([prompt, gen], 1), prefix_len=L)[:, -1]
                else:
                    logits = model(torch.cat([prompt, gen], 1))[:, -1]
            nxt = sample_next(logits, gen, temperature, top_k, top_p, rep_penalty, banned)
            gen = torch.cat([gen, nxt[:, None]], 1)
            if (ws[gen].sum(1) > target_words).all():      # 129 word starts => word 128 is complete
                break
        for j, i in enumerate(bidx):
            words = tok.decode(gen[j].tolist()).split()[:target_words]
            out[i] = " ".join(words) if words else "the"
        if log_every and bi % log_every == 0:
            print(f"  generated {min(b0 + batch_size, len(order))}/{len(order)}", flush=True)
    return out

def write_submission(preds, ids, json_path, zip_path=None):
    missing = [i for i in ids if i not in preds]
    assert not missing, f"{len(missing)} ids missing"
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump({i: {"text": preds[i]} for i in ids}, f, ensure_ascii=False, indent=2)
    if zip_path:
        with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
            z.write(json_path, arcname="prediction.json")      # file at the archive root

## B5. `models.py` — YOUR architectures
Course rule §17.5: the model architecture code must be written by you. This stub fixes the **interface** the pipeline needs and describes **each step in plain English**; you write the lines marked `TODO(you)`.
Write `DecoderLM` first, run B6 to check it, then the other three: each one is DecoderLM with one thing changed.

In [ ]:
%%writefile models.py
# =====================================================================================================
#  CS-781 Phase 1 — four language models                      
# =====================================================================================================

import math
import torch
import torch.nn as nn


# =====================================================================================================
#  PART 1 — MASKS
# =====================================================================================================

def causal_mask(T, device):
    mask = torch.ones(T, T, dtype=torch.bool, device=device).triu(diagonal=1)
    return mask


def prefix_mask(T, P, device):
    mask = causal_mask(T, device)
    mask[:, :P] = False
    return mask


def alibi_window_bias(T, n_heads, window, device):
    i = torch.arange(T, device=device)[:, None]
    j = torch.arange(T, device=device)[None, :]

    dist = i - j

    slopes = torch.tensor(
        [2 ** (-8 * (h + 1) / n_heads) for h in range(n_heads)],
        device=device,
        dtype=torch.float32,
    )[:, None, None]

    bias = -slopes * dist

    bias = bias.masked_fill(
        (dist < 0) | (dist >= window),
        float("-inf")
    )

    return bias


# =====================================================================================================
#  PART 2 — SHARED HELPERS
# =====================================================================================================

def make_layers(n_layers, d_model, n_heads, dropout):

    return nn.ModuleList([
        nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=4 * d_model,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        for _ in range(n_layers)
    ])


def init_weights(module):
    if isinstance(module, (nn.Linear, nn.Embedding)):
        nn.init.normal_(module.weight, mean=0.0, std=0.02)

    if isinstance(module, nn.Linear) and module.bias is not None:
        nn.init.zeros_(module.bias)


# =====================================================================================================
#  MODEL 1 — DECODER-ONLY (GPT-style)
# =====================================================================================================

class DecoderLM(nn.Module):
    kind = "causal"

    def __init__(self, vocab_size, d_model, n_layers, n_heads, max_len, dropout=0.1):
        super().__init__()

        self.max_len = max_len

        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)

        self.layers = make_layers(
            n_layers,
            d_model,
            n_heads,
            dropout
        )

        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)

        self.apply(init_weights)

        # Weight tying
        self.head.weight = self.tok_emb.weight

    def forward(self, idx):
        B, T = idx.shape

        pos = torch.arange(T, device=idx.device)

        x = self.tok_emb(idx) + self.pos_emb(pos)

        mask = causal_mask(T, idx.device)

        for layer in self.layers:
            x = layer(x, src_mask=mask)

        return self.head(self.norm(x))


# =====================================================================================================
#  MODEL 2 — PREFIX-LM
# =====================================================================================================

class PrefixLM(nn.Module):
    kind = "prefix"

    def __init__(self, vocab_size, d_model, n_layers, n_heads, max_len, dropout=0.1):
        super().__init__()

        self.max_len = max_len

        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)

        self.layers = make_layers(
            n_layers,
            d_model,
            n_heads,
            dropout
        )

        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)

        self.apply(init_weights)

        # Weight tying
        self.head.weight = self.tok_emb.weight

    def forward(self, idx, prefix_len):
        B, T = idx.shape

        pos = torch.arange(T, device=idx.device)

        x = self.tok_emb(idx) + self.pos_emb(pos)

        mask = prefix_mask(T, prefix_len, idx.device)

        for layer in self.layers:
            x = layer(x, src_mask=mask)

        return self.head(self.norm(x))


# =====================================================================================================
#  MODEL 3 — ENCODER-DECODER
# =====================================================================================================

class EncDecLM(nn.Module):
    kind = "encdec"

    def __init__(
        self,
        vocab_size,
        d_model,
        n_enc_layers,
        n_dec_layers,
        n_heads,
        max_len,
        max_tgt_len,
        dropout=0.1,
    ):
        super().__init__()

        self.max_len, self.max_tgt_len = max_len, max_tgt_len

        # Shared token embedding
        self.tok_emb = nn.Embedding(vocab_size, d_model)

        # Separate position embeddings
        self.src_pos_emb = nn.Embedding(max_len, d_model)
        self.tgt_pos_emb = nn.Embedding(max_tgt_len, d_model)

        # Encoder
        self.enc_layers = make_layers(
            n_enc_layers,
            d_model,
            n_heads,
            dropout
        )

        self.enc_norm = nn.LayerNorm(d_model)

        # Decoder
        self.dec_layers = nn.ModuleList([
            nn.TransformerDecoderLayer(
                d_model=d_model,
                nhead=n_heads,
                dim_feedforward=4 * d_model,
                dropout=dropout,
                activation="gelu",
                batch_first=True,
                norm_first=True,
            )
            for _ in range(n_dec_layers)
        ])

        self.dec_norm = nn.LayerNorm(d_model)

        self.head = nn.Linear(d_model, vocab_size, bias=False)

        self.apply(init_weights)

        # Weight tying
        self.head.weight = self.tok_emb.weight

    def forward(self, src, tgt_in):
        S, U = src.size(1), tgt_in.size(1)

        # -------------------------
        # ENCODER
        # -------------------------

        src_pos = torch.arange(S, device=src.device)

        mem = self.tok_emb(src) + self.src_pos_emb(src_pos)

        for layer in self.enc_layers:
            mem = layer(mem)

        mem = self.enc_norm(mem)

        # -------------------------
        # DECODER
        # -------------------------

        tgt_pos = torch.arange(U, device=tgt_in.device)

        y = self.tok_emb(tgt_in) + self.tgt_pos_emb(tgt_pos)

        tgt_mask = causal_mask(U, tgt_in.device)

        for layer in self.dec_layers:
            y = layer(
                y,
                mem,
                tgt_mask=tgt_mask
            )

        return self.head(self.dec_norm(y))


# =====================================================================================================
#  MODEL 4 — EFFICIENT: SLIDING-WINDOW ATTENTION + ALiBi
# =====================================================================================================

class SlidingWindowLM(nn.Module):
    kind = "causal"

    def __init__(
        self,
        vocab_size,
        d_model,
        n_layers,
        n_heads,
        window,
        max_len,
        dropout=0.1,
    ):
        super().__init__()

        self.max_len, self.n_heads, self.window = max_len, n_heads, window

        self.tok_emb = nn.Embedding(vocab_size, d_model)

        self.layers = make_layers(
            n_layers,
            d_model,
            n_heads,
            dropout
        )

        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)

        self.apply(init_weights)

        # Weight tying
        self.head.weight = self.tok_emb.weight

    def forward(self, idx):
        B, T = idx.shape

        # Token embeddings only
        x = self.tok_emb(idx)

        # ALiBi + sliding-window bias
        bias = alibi_window_bias(
            T,
            self.n_heads,
            self.window,
            idx.device
        )

        # (n_heads, T, T)
        # -> (B * n_heads, T, T)
        bias = bias.repeat(B, 1, 1)

        for layer in self.layers:
            x = layer(
                x,
                src_mask=bias
            )

        return self.head(self.norm(x))


# =====================================================================================================
#  FACTORY — do not change.
# =====================================================================================================

def build_model(arch, vocab_size, **cfg):
    classes = {
        "decoder": DecoderLM,
        "prefix": PrefixLM,
        "encdec": EncDecLM,
        "swa": SlidingWindowLM
    }

    return classes[arch](vocab_size, **cfg)

## B6. Size and speed check (run interactively before committing, ≈5 min GPU)
All four lines must say **OK** and fit well under 15 GB. The initial loss should be close to ln(V). "Tokens in budget" shows how much data each model sees in `hours_per_model`: bigger models learn more per token but see fewer tokens, and that trade-off is yours. Set `CHECK_ARCHS` to only the architectures you have finished.

In [ ]:
import json, time, importlib, torch, numpy as np
import lm_common, models
importlib.reload(models)

CHECK_ARCHS = ALL_ARCHS          # e.g. ["decoder"] while the others are still TODO
C = json.load(open("/kaggle/working/run_config.json")); T_ = C["train"]
tok = lm_common.Tok()
data = np.memmap(lm_common.find_input("train.bin"), dtype=np.uint16, mode="r")
print(f"vocab {tok.V} | ln(V) = {np.log(tok.V):.2f} (initial loss should be close to this)")
for arch in CHECK_ARCHS:
    m = models.build_model(arch, vocab_size=tok.V, **C["models"][arch]).cuda()
    n = sum(p.numel() for p in m.parameters())
    opt, scaler = torch.optim.AdamW(m.parameters(), lr=1e-4), torch.cuda.amp.GradScaler()
    torch.cuda.reset_peak_memory_stats()
    for i in range(13):
        if i == 3:
            torch.cuda.synchronize(); t0 = time.time()
        seq = lm_common.get_batch(data, T_["batch_size"], T_["block_size"], "cuda")
        with torch.autocast("cuda", dtype=torch.float16):
            loss = lm_common.compute_loss(m, seq, tok.bos)
        if i == 0:
            first_loss = loss.item()
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
    torch.cuda.synchronize()
    tps = T_["batch_size"] * T_["block_size"] / ((time.time() - t0) / 10)
    print(f"{arch:8s} {n/1e6:6.1f}M params [{'OK' if 50e6 <= n <= 300e6 else 'OUT OF RANGE'}] | "
          f"{tps:,.0f} tok/s | peak {torch.cuda.max_memory_allocated()/1e9:.1f} GB | "
          f"~{tps * C['hours_per_model'] * 3600 / 1e6:,.0f}M tokens in budget | "
          f"initial loss {first_loss:.2f} -> {loss.item():.2f}")
    del m, opt; torch.cuda.empty_cache()

## B7. Training script (trains for `hours_per_model`, keeps the best checkpoint, then writes validation + test predictions and the submission zip)

In [ ]:
%%writefile train.py
import argparse, json, math, os, random, time
import numpy as np, torch
from lm_common import Tok, find_input, get_batch, compute_loss, eval_loss, generate, write_submission
import models

ap = argparse.ArgumentParser()
ap.add_argument("--arch", required=True)
ap.add_argument("--gpu", type=int, default=0)
args = ap.parse_args()

C = json.load(open("/kaggle/working/run_config.json"))
T_, G, arch = C["train"], C["gen"], args.arch
dev = f"cuda:{args.gpu}"; torch.cuda.set_device(dev)
random.seed(C["seed"]); np.random.seed(C["seed"]); torch.manual_seed(C["seed"])
OUT = f"/kaggle/working/runs/{arch}"; os.makedirs(OUT, exist_ok=True)
t_start = time.time()
log = open(f"{OUT}/log.jsonl", "a")
def say(**kw):
    kw["t"] = round(time.time() - t_start); log.write(json.dumps(kw) + "\n"); log.flush()
    print(arch, kw, flush=True)

tok = Tok()
train_data = np.memmap(find_input("train.bin"), dtype=np.uint16, mode="r")
val_data = np.memmap(find_input("val.bin"), dtype=np.uint16, mode="r")
model = models.build_model(arch, vocab_size=tok.V, **C["models"][arch]).to(dev)
n_params = sum(p.numel() for p in model.parameters())
assert 50e6 <= n_params <= 300e6, f"{arch}: {n_params/1e6:.1f}M params is outside 50M-300M"
block = T_["block_size"]
if model.kind == "encdec":
    assert model.max_len + model.max_tgt_len >= block + 1
else:
    assert model.max_len >= block

decay = [p for p in model.parameters() if p.dim() >= 2]
no_decay = [p for p in model.parameters() if p.dim() < 2]
opt = torch.optim.AdamW([{"params": decay, "weight_decay": T_["weight_decay"]},
                         {"params": no_decay, "weight_decay": 0.0}], lr=T_["lr"], betas=tuple(T_["betas"]))
scaler = torch.cuda.amp.GradScaler()
budget = C["hours_per_model"] * 3600

def lr_at(step, frac):                     # linear warm-up, then cosine over the TIME budget
    if step < T_["warmup_steps"]:
        return T_["lr"] * (step + 1) / T_["warmup_steps"]
    return T_["min_lr"] + 0.5 * (T_["lr"] - T_["min_lr"]) * (1 + math.cos(math.pi * min(frac, 1.0)))

def save(path, step, vl):
    torch.save({"model": model.state_dict(), "arch": arch, "cfg": C["models"][arch],
                "step": step, "val_loss": vl}, path)

say(event="start", params=n_params, kind=model.kind)
step, tokens, best = 0, 0, float("inf")
model.train()
while (frac := (time.time() - t_start) / budget) < 1:
    lr = lr_at(step, frac)
    for g in opt.param_groups:
        g["lr"] = lr
    tot = 0.0
    for _ in range(T_["grad_accum"]):
        seq = get_batch(train_data, T_["batch_size"], block, dev)
        with torch.autocast("cuda", dtype=torch.float16):
            loss = compute_loss(model, seq, tok.bos) / T_["grad_accum"]
        scaler.scale(loss).backward(); tot += loss.item()
    scaler.unscale_(opt)
    torch.nn.utils.clip_grad_norm_(model.parameters(), T_["grad_clip"])
    scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
    step += 1; tokens += T_["batch_size"] * T_["grad_accum"] * block
    if step % T_["log_every"] == 0:
        say(event="train", step=step, loss=round(tot, 4), lr=lr, tokens=tokens)
    if step % T_["eval_every"] == 0:
        vl = eval_loss(model, val_data, T_["batch_size"], block, dev, tok.bos)
        say(event="eval", step=step, val_loss=round(vl, 4), tokens=tokens)
        if vl < best:
            best = vl; save(f"{OUT}/best.pt", step, vl)

vl = eval_loss(model, val_data, T_["batch_size"], block, dev, tok.bos)
say(event="eval", step=step, val_loss=round(vl, 4), tokens=tokens)
if vl < best:
    best = vl; save(f"{OUT}/best.pt", step, vl)

# ---- generate with the best checkpoint ----
model.load_state_dict(torch.load(f"{OUT}/best.pt", map_location=dev)["model"])
del opt; torch.cuda.empty_cache()
vp = json.load(open(find_input("val_prompts.json"))); vids = list(vp)
preds = generate(model, tok, [vp[i]["text"] for i in vids], dev, **G)
write_submission(dict(zip(vids, preds)), vids, f"{OUT}/val_prediction.json")
tp = json.load(open("/kaggle/working/test.json")); tids = list(tp)
preds = generate(model, tok, [tp[i]["text"] for i in tids], dev, **G)
write_submission(dict(zip(tids, preds)), tids, f"{OUT}/prediction.json", f"{OUT}/submission_{arch}.zip")
say(event="done", best_val_loss=best, steps=step, tokens=tokens)

## B8. Launch: one model per GPU, both in parallel; prints progress every 10 min

In [ ]:
import subprocess, sys, time, torch

n_gpu = torch.cuda.device_count()
assert n_gpu >= 1
waves = [SESSION_ARCHS[i:i + n_gpu] for i in range(0, len(SESSION_ARCHS), n_gpu)]   # 2 GPUs -> one wave
for wave in waves:
    procs = []
    for gpu, arch in enumerate(wave):
        out = open(f"/kaggle/working/{arch}.out", "w")
        procs.append((arch, subprocess.Popen([sys.executable, "train.py", "--arch", arch, "--gpu", str(gpu)],
                                             stdout=out, stderr=subprocess.STDOUT, cwd="/kaggle/working")))
    while any(p.poll() is None for _, p in procs):
        time.sleep(600)
        for arch, _ in procs:
            lines = open(f"/kaggle/working/{arch}.out").read().strip().splitlines()
            print(time.strftime("%H:%M"), "|", lines[-1][:200] if lines else "(no output yet)", flush=True)
    for arch, p in procs:
        print(f"{arch}: exit code {p.returncode}")
        if p.returncode != 0:
            print(open(f"/kaggle/working/{arch}.out").read()[-3000:])

## B9. Loss curves (for the report)

In [ ]:
import json, glob
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
for path in sorted(glob.glob("/kaggle/working/runs/*/log.jsonl")):
    arch = path.split("/")[-2]
    rows = [json.loads(l) for l in open(path)]
    tr = [r for r in rows if r["event"] == "train"]; ev = [r for r in rows if r["event"] == "eval"]
    ax[0].plot([r["tokens"] / 1e6 for r in tr], [r["loss"] for r in tr], label=arch, alpha=.8)
    ax[1].plot([r["tokens"] / 1e6 for r in ev], [r["val_loss"] for r in ev], "o-", label=arch)
for a, t in zip(ax, ["Training loss", "Validation loss"]):
    a.set(title=t, xlabel="Training tokens (M)", ylabel="Cross-entropy"); a.grid(alpha=.3); a.legend()
plt.tight_layout(); plt.savefig("/kaggle/working/loss_curves.png", dpi=200); plt.show()

## B10. Official local evaluation on your validation set
The first cell is the course's `evaluate_phase1.py`, copied unchanged from the evaluation-scripts zip. Note: validation loss of `prefix` and `encdec` is computed only on the continuation part, so compare architectures with **BERTScore**, not with validation loss.

In [ ]:
%%writefile evaluate_phase1.py
#!/usr/bin/env python3
"""Local evaluator for Phase 1.

Usage:
  python evaluate_phase1.py --truth truth.json --prediction prediction.json

Requires: bert-score and torch for the official primary metric.
"""
import argparse, json, math, sys
from collections import Counter

try:
    import bert_score
except ImportError as exc:
    raise SystemExit("Missing dependency: bert-score. Install it from requirements.docx / requirements.txt.") from exc

class DuplicateJSONKeyError(ValueError): pass

def reject_constant(v): raise ValueError(f"Non-standard JSON numeric value '{v}' is not allowed.")

def pairs_no_duplicates(pairs):
    out = {}; dup = []
    for k, v in pairs:
        if k in out: dup.append(k)
        out[k] = v
    if dup: raise DuplicateJSONKeyError(f"Duplicate JSON object keys detected: {list(dict.fromkeys(dup))[:10]}.")
    return out

def load_json(path):
    try:
        with open(path, encoding='utf-8') as f:
            return json.load(f, object_pairs_hook=pairs_no_duplicates, parse_constant=reject_constant)
    except json.JSONDecodeError as e:
        raise ValueError(f"Invalid JSON in {path} at line {e.lineno}, column {e.colno}: {e.msg}") from e

def validate(data, name):
    if not isinstance(data, dict) or not data: raise ValueError(f"{name} must be a non-empty JSON object mapping IDs to records.")
    for i, rec in data.items():
        if not isinstance(i, str) or not i.strip() or not isinstance(rec, dict) or 'text' not in rec or not isinstance(rec['text'], str) or not rec['text'].strip():
            raise ValueError(f"Invalid record for ID {i!r} in {name}; each record must contain non-empty string field 'text'.")

def lcs_length(a,b):
    if not a or not b: return 0
    if len(a) < len(b): a,b=b,a
    masks={}
    for j,t in enumerate(a): masks[t]=masks.get(t,0)|(1<<j)
    state=0
    for t in b:
        m=masks.get(t,0); x=m|state; y=(state<<1)|1; state=x & ~(x-y)
    return state.bit_count() if hasattr(int,'bit_count') else bin(state).count('1')

def rouge_l_f1(p,r):
    pt=p.split(); rt=r.split()
    if not pt or not rt: return 0.0
    l=lcs_length(pt,rt)
    if not l: return 0.0
    pp=l/len(pt); rr=l/len(rt)
    return 0.0 if pp+rr==0 else 2*pp*rr/(pp+rr)

def ngrams(t,n): return Counter(tuple(t[i:i+n]) for i in range(len(t)-n+1)) if len(t)>=n else Counter()

def bleu4(preds, refs):
    order=4; clipped=[0]*order; totals=[0]*order; pl=rl=0
    for p,r in zip(preds,refs):
        pl+=len(p); rl+=len(r)
        for n in range(1,order+1):
            pn=ngrams(p,n); rn=ngrams(r,n); totals[n-1]+=sum(pn.values())
            for g,c in pn.items(): clipped[n-1]+=min(c,rn.get(g,0))
    prec=[clipped[i]/totals[i] if totals[i] else 0.0 for i in range(order)]
    if any(x<=0 for x in prec) or pl==0: return 0.0
    bp=1.0 if pl>rl else math.exp(1.0-rl/pl)
    return bp*math.exp(sum(math.log(x) for x in prec)/order)

def main():
    ap=argparse.ArgumentParser(description='Evaluate Phase 1 predictions on your own held-out split.')
    ap.add_argument('--truth', required=True, help='Your held-out reference JSON.')
    ap.add_argument('--prediction', required=True, help='Your prediction JSON.')
    args=ap.parse_args()
    truth=load_json(args.truth); pred=load_json(args.prediction)
    validate(truth,'truth'); validate(pred,'prediction')
    if set(truth)!=set(pred): raise ValueError(f"ID mismatch: missing={len(set(truth)-set(pred))}, extra={len(set(pred)-set(truth))}")
    ps=[pred[i]['text'].split() for i in truth]; rs=[truth[i]['text'].split() for i in truth]
    pred_texts=[pred[i]['text'].strip() for i in truth]; ref_texts=[truth[i]['text'].strip() for i in truth]
    try:
        import torch
        device='cuda' if torch.cuda.is_available() else 'cpu'
    except ImportError: device=None
    _,_,f=bert_score.score(pred_texts, ref_texts, lang='en', batch_size=8, verbose=False, **({'device':device} if device else {}))
    bertscore=float(f.mean().item())
    rouge=sum(rouge_l_f1(pred[i]['text'],truth[i]['text']) for i in truth)/len(truth)
    b4=bleu4(ps,rs)
    print(f'Examples: {len(truth)}')
    print(f'BERTScore F1: {bertscore:.6f}')
    print(f'ROUGE-L F1:   {rouge:.6f}')
    print(f'BLEU-4:       {b4:.6f}')
if __name__=='__main__':
    try: main()
    except Exception as e: print(f'EVALUATION FAILED: {e}', file=sys.stderr); sys.exit(1)

In [ ]:
!pip install -q bert-score
import os, lm_common
VAL_TRUTH = lm_common.find_input("val_truth.json")
for arch in sorted(os.listdir("/kaggle/working/runs")):
    pred = f"/kaggle/working/runs/{arch}/val_prediction.json"
    if os.path.exists(pred):
        print("=====", arch)
        !python evaluate_phase1.py --truth {VAL_TRUTH} --prediction {pred}

## B11. Check the submission zips before uploading to Codabench

In [ ]:
import zipfile, json, glob
test_ids = set(json.load(open("/kaggle/working/test.json")))
for z in sorted(glob.glob("/kaggle/working/runs/*/submission_*.zip")):
    with zipfile.ZipFile(z) as zf:
        names, pred = zf.namelist(), json.loads(zf.read("prediction.json"))
    wc = [len(v["text"].split()) for v in pred.values()]
    ok = names == ["prediction.json"] and set(pred) == test_ids and all(v["text"].strip() for v in pred.values())
    print(f"{z}: {'OK' if ok else 'PROBLEM'} | files={names} | ids={len(pred)} | "
          f"words min/mean/max = {min(wc)}/{sum(wc)/len(wc):.1f}/{max(wc)}")

## B12. (Optional) Compare decoding settings on a trained model → extra, genuinely different submissions
To use session-1 models during session 2, add session 1's notebook version as an extra input.

In [ ]:
import json, torch, lm_common, models
ARCH = "decoder"
SETTINGS = {            # YOUR candidate decoding settings
    "a": dict(temperature=0.1, top_k=0, top_p=0.95, rep_penalty=1.0),
}
C = json.load(open("/kaggle/working/run_config.json"))
tok = lm_common.Tok()
ck = torch.load(lm_common.find_input(f"{ARCH}/best.pt"), map_location="cuda:0")
model = models.build_model(ARCH, vocab_size=tok.V, **ck["cfg"]).to("cuda:0")
model.load_state_dict(ck["model"])
vp = json.load(open(lm_common.find_input("val_prompts.json"))); vids = list(vp)
truth = lm_common.find_input("val_truth.json")
for name, s in SETTINGS.items():
    preds = lm_common.generate(model, tok, [vp[i]["text"] for i in vids], "cuda:0", **{**C["gen"], **s})
    path = f"/kaggle/working/sweep_{ARCH}_{name}.json"
    lm_common.write_submission(dict(zip(vids, preds)), vids, path)
    print("=====", name, s)
    !python evaluate_phase1.py --truth {truth} --prediction {path}

# To make a test submission with the setting you chose:
# best = SETTINGS["a"]; tp = json.load(open("/kaggle/working/test.json")); tids = list(tp)
# preds = lm_common.generate(model, tok, [tp[i]["text"] for i in tids], "cuda:0", **{**C["gen"], **best})
# lm_common.write_submission(dict(zip(tids, preds)), tids, "/kaggle/working/prediction.json",
#                            f"/kaggle/working/submission_{ARCH}_a.zip")